# NB-06-02 — Regression and Classification Workbench

**Module:** M06 · **Time:** 60–90 minutes · **Prerequisites:** L06-03 to L06-06

Fit the same data with a closed-form solution, gradient descent, polynomial terms, ridge and
lasso; then fit a classifier, choose a threshold, and evaluate it with a confusion matrix and ROC.
Every number here appears in the lessons.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

for parent in [Path.cwd(), *Path.cwd().parents]:
    candidate = parent / "modules" / "M06-machine-learning-fundamentals" / "code"
    if candidate.is_dir():
        sys.path.insert(0, str(candidate))
        break
print("code directory:", candidate)

In [ ]:
import numpy as np
import datasets as ds
import linreg as lr
import evaluation as ev
from prep import Pipeline, StandardScaler, SimpleImputer, train_test_split
from classifiers import LogisticRegressionGD

## 2. OLS versus the mean baseline

In [ ]:
frame = ds.students()
cols = ["study_hours", "attendance_pct", "prior_grade", "sleep_hours"]
X = frame[cols].fillna(frame[cols].median()).to_numpy()
y = frame["final_score"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, 0.25, seed=0)

model = lr.LinearRegression().fit(X_train, y_train)
print("coefficients:", dict(zip(cols, np.round(model.coef_, 3))),
      "intercept", round(model.intercept_, 2))
print({k: round(v, 3) for k, v in lr.regression_report(y_test, model.predict(X_test), 4).items()})

baseline = np.full_like(y_test, y_train.mean())
print("baseline MAE:", round(lr.mae(y_test, baseline), 3),
      "R2:", round(lr.r2_score(y_test, baseline), 3))

## 3. Gradient descent and scaling

In [ ]:
raw = lr.LinearRegressionGD(lr=0.1, epochs=50).fit(X_train, y_train)
scaled = lr.LinearRegressionGD(lr=0.1, epochs=500, standardize=True).fit(X_train, y_train)
print("raw loss first/last :", round(raw.loss_history_[0], 1), raw.loss_history_[-1])
print("scaled loss first/last:", round(scaled.loss_history_[0], 1), round(scaled.loss_history_[-1], 2))
print("scaled test R2:", round(scaled.score(X_test, y_test), 4))

The raw run diverges because the learning rate is too large for the feature scales. Both
start at the same loss; only the conditioning differs.

## 4. Polynomial capacity

In [ ]:
for degree in range(1, 6):
    poly = lr.PolynomialFeatures(degree).fit(X_train)
    m = lr.LinearRegression().fit(poly.transform(X_train), y_train)
    print(f"degree {degree}: {poly.n_output_features_:3d} columns "
          f"train {m.score(poly.transform(X_train), y_train):.3f} "
          f"test {m.score(poly.transform(X_test), y_test):.3f}")

## 5. Ridge and lasso

In [ ]:
for alpha in [0.0, 1.0, 10.0, 100.0]:
    ridge = lr.Ridge(alpha).fit(X_train, y_train)
    lasso = lr.Lasso(alpha if alpha else 1e-8).fit(X_train, y_train)
    print(f"alpha {alpha:6.1f} ridge {ridge.score(X_test, y_test):.4f} "
          f"lasso {lasso.score(X_test, y_test):.4f} "
          f"nonzero {int(np.sum(np.abs(lasso.coef_) > 1e-8))}")

## 6. A classifier, end to end

In [ ]:
loans = ds.loans()
lcols = ["annual_income_k", "loan_amount_k", "credit_score",
         "employment_years", "debt_ratio", "default_history"]
Xl = loans[lcols].fillna(loans[lcols].median()).to_numpy()
yl = loans["approved"].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(Xl, yl, 0.25, seed=0, stratify=yl)
scaler = StandardScaler().fit(Xtr)
clf = LogisticRegressionGD(lr=0.5, epochs=800).fit(scaler.transform(Xtr), ytr)
proba = clf.predict_proba(scaler.transform(Xte))[:, 1]
pred = (proba >= 0.5).astype(int)

print(ev.confusion_matrix(yte, pred))
print(ev.classification_report(yte, pred).round(3).to_string(index=False))
print("accuracy", round(ev.accuracy(yte, pred), 4),
      "AUC", round(ev.roc_auc_score(yte, proba), 4),
      "AP", round(ev.average_precision(yte, proba), 4))

## 7. The threshold is a decision

In [ ]:
print(ev.threshold_sweep(yte, proba, [0.1, 0.2, 0.3, 0.4, 0.5]).round(3).to_string(index=False))

# expected cost if a false approval costs 10 and a false rejection costs 1
sweep = ev.threshold_sweep(yte, proba, np.round(np.arange(0.05, 0.96, 0.05), 2))
sweep["cost"] = 10 * (sweep["flagged"] - sweep["recall"] * yte.sum()) + 1 * ((1 - sweep["recall"]) * yte.sum())
best = sweep.loc[sweep["cost"].idxmin()]
print("cost-optimal threshold:", best["threshold"], "cost:", round(best["cost"], 1))

## 8. Cross-validation and an interval

In [ ]:
def make():
    return Pipeline([("impute", SimpleImputer("median")),
                     ("scale", StandardScaler()),
                     ("model", LogisticRegressionGD(lr=0.5, epochs=400))])

cv = ev.cross_val_score(make, Xl, yl, k=5, stratified=True)
print("CV:", np.round(cv, 4), "mean", round(cv.mean(), 4), "std", round(cv.std(), 4))
print("bootstrap:", {k: round(v, 3) for k, v in ev.bootstrap_ci(yte, pred).items()})

## 9. Reflect

1. Which model would you ship for each of the two problems, and on which metric?
2. The polynomial and the lasso both have knobs. How do you choose them without touching test?
3. Why does the cost-optimal threshold differ from the F1-optimal one?

Next: **NB-06-03** for the model comparison.